<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma3:12b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

In the previous notebook, we added a basic `Memory` module that simply tracks the conversation history to the chapter, we covered which LLM to choose using various inference engines. In this notebook we will cover more forms of short-term memory


## 2 - Adding **`TrimmingMemory`**

The first technique for efficient short-term memory is rather straightforward: trimming the messages as they grow. Whenever the number of messages grows too large for the LLM’s context window to handle, we can decide to simply remove the first few interactions until it fits that window. 

Since we already did much of the heavy lifting with the `Memory` module, implementing this trimming behavior requires minimal amount of code. We use inheritance to keep the same functionality of Memory but update the add function so that only the last two turns (each a pair of user/assisant messages) are kept:


The issue with the `TinyAgent` that we have thus far, is that it does not track and remember its previous conversations, it is stateless. Let us demonstrate with an example by using the Agent from Chapter 2:

In [3]:
from illustrated_agents.chapters.ch4 import Memory

class TrimmingMemory(Memory):
    """Memory that keeps only the last two user/assistant turns."""

    def add(self, role: str, content: str) -> None:
        # Add the new message first using the parent class (Memory)
        super().add(role, content)

        # Then, keep system message plus the most recent two turns (4 messages)
        system = [message for message in self.messages if message["role"] == "system"]
        turns = [message for message in self.messages if message["role"] != "system"]
        self.messages = system + turns[-4:]

Let's try it out with a couple of message and see if the Agent correctly kept only the last two turns.

In [8]:
from illustrated_agents.chapters.ch4 import TinyAgent

# Create the Agent
memory = TrimmingMemory()
agent = TinyAgent(llm=llm, memory=memory)

# Run some interactions
response_1 = agent.run("Hi! I'm reading 'An Illustrated Guide to AI Agents'.")
response_2 = agent.run("There are many flamingos in this book, why?")
response_3 = agent.run("What is 1+1?")
response_4 = agent.run("What is 2+2?")

We can then inspect the memory as we did before:

In [9]:
from rich import print
print(agent.memory.get_messages())

[
    {'role': 'user', 'content': 'What is 1+1?'},
    {
        'role': 'assistant',
        'content': "1 + 1 = 2\n\nIt's a classic! 😊\n\n\n\nAre you sure you just wanted to check that, or were you 
testing me? 😉"
    },
    {'role': 'user', 'content': 'What is 2+2?'},
    {'role': 'assistant', 'content': '2 + 2 = 4! \n\n\n\nJust keeping the math train rolling, I see! 😉'}
]

Note how only the last two interactions are tracked in the memory. That helps to keep the memory minimal but may remove important information early on in the process.

Note that we can also check the Trajectory, which should have the full conversation:

In [7]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

## 3 - Adding **`SummarizationMemory`**
A common technique is to employ another LLM to summarize the conversation history. After each conversation turn, the same or another LLM will summarize it and add it to the full summary of the conversation.

Summarization can be achieved in many different ways, like summarizing the entire conversation history or only parts of it. In this example, we are going to be using the same LLM to summarize the entire conversation history. We will use the `system` role to track the summary to we can separate it from the conversation between the `user` and `assistant`.

In [61]:
class SummarizationMemory(Memory):
    """Memory that compresses past turns into a running summary via an LLM."""

    def __init__(self, llm: LLM):
        super().__init__()
        self.llm = llm

    def add(self, role: str, content: str) -> None:
        # Add the new message first using the parent class (Memory)
        super().add(role, content)

        # After each completed turn, update the running summary
        if role == "assistant":
            # Split the existing summary from the new conversation turns
            summary = ""
            conversation = ""
            for message in self.messages:
                if message["role"] == "system":
                    summary = message["content"]
                else:
                    conversation += f"{message['role']}: {message['content']}\n"

            # Ask the LLM to extend the summary with the new conversation
            prompt = f"""Update the summary with the new conversation.

Summary: {summary}

Conversation:
{conversation}
Output the updated summary only."""
            response = self.llm.generate([{"role": "user", "content": prompt}])
            self.messages = [{"role": "system", "content": response.content}]

We can use this `SummarizationMemory` as follows:

In [53]:
# Create the Agent
memory = SummarizationMemory(llm=llm)
agent = TinyAgent(llm=llm, memory=memory)

# Run a query
response = agent.run("Hi, my name is Sarah. Why are flamingos pink?")
print(response)

Hi Sarah! That's a great question! Flamingos aren't born pink – they're actually born with grey or white feathers. 
They turn pink because of their diet!

Here's the breakdown:

*   **They eat shrimp, algae, and brine flies:** These foods contain pigments called carotenoids. Carotenoids are 
also what make carrots orange and tomatoes red.
*   **Carotenoids are absorbed into their bodies:** As flamingos digest these foods, their bodies absorb the 
carotenoids.
*   **The pigments deposit in their feathers:**  These pigments are then deposited in their feathers, gradually 
turning them pink.

The more carotenoid-rich food a flamingo eats, the pinker it will become! Some flamingos have a very vibrant pink 
color, while others are paler.



Do you have any other questions about flamingos, or anything else you're curious about?

The messages in the memory module are now updated so that only the summary is tracked in the `system` role:

In [54]:
print(agent.memory.get_messages())

[
    {
        'role': 'system',
        'content': 'Sarah asked why flamingos are pink. The assistant explained that flamingos are not born pink; 
they are born with grey or white feathers. They turn pink due to carotenoid pigments found in their diet of shrimp,
algae, and brine flies. These pigments are absorbed and deposited into their feathers, causing the pink coloration.
The intensity of the pink depends on the amount of carotenoid-rich food they consume.'
    }
]

We can check what happens with the summary if we run the Agent a couple of times.

In [63]:
response = agent.run("Tell me about yourself.")
response = agent.run("What is your favorite color?")
response = agent.run("Are you fully autonomous?")
print(agent.memory.get_messages())

[
    {
        'role': 'system',
        'content': "Summary: Sarah asked why flamingos are pink. The assistant explained that flamingos are not 
born pink; they are born with grey or white feathers. They turn pink due to carotenoid pigments found in their diet
of shrimp, algae, and brine flies. These pigments are absorbed and deposited into their feathers, causing the pink 
coloration. The intensity of the pink depends on the amount of carotenoid-rich food they consume. Following this, 
the user asked the assistant to describe itself. The assistant responded that it is a large language model, trained
by Google, designed to understand and generate human language, answer questions, generate creative text formats, 
and perform tasks like translation and summarization. It clarified that it lacks personal experiences or emotions, 
does not have a body, and cannot provide real-time information or professional advice; it's a helpful tool, not a 
substitute for a human expert. Subsequently, the user inquired about the assistant's favorite color. The assistant 
responded that it cannot have a favorite color because it does not experience emotions or possess senses like 
sight. The user then asked if the assistant was fully autonomous. The assistant responded that it is not, 
explaining it operates based on its training data and programming, requires prompts to generate responses, lacks 
independent decision-making capabilities, and is controlled by Google. It emphasized that it is a sophisticated 
tool responding to instructions, not a self-governing entity. According to the provided text, Sarah is the user's 
name."
    }
]

Note how the summary now includes the other questions we asked the model!

As always, to see the full trajectory:

In [64]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [ ]:
from illustrated_agents.chapters.ch4_short_term_memory import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py  ← Updated (Integrated `Memory` into your `TinyAgent`.)                                            │
│ ├── llm.py                                                                                                      │
│ └── memory.py ← New (Track conversation history.)                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯